### Check Conda Environment

In [1]:
from __future__ import print_function
from packaging.version import parse as Version
from platform import python_version

OK = '\x1b[42m[ OK ]\x1b[0m'
FAIL = "\x1b[41m[FAIL]\x1b[0m"

try:
    import importlib
except ImportError:
    print(FAIL, "Python version 3.12.10 is required,"
                " but %s is installed." % sys.version)

def import_version(pkg, min_ver, fail_msg=""):
    mod = None
    try:
        mod = importlib.import_module(pkg)
        if pkg in {'PIL'}:
            ver = mod.VERSION
        else:
            ver = mod.__version__
        if Version(ver) == Version(min_ver):
            print(OK, "%s version %s is installed."
                  % (lib, min_ver))
        else:
            print(FAIL, "%s version %s is required, but %s installed."
                  % (lib, min_ver, ver))    
    except ImportError:
        print(FAIL, '%s not installed. %s' % (pkg, fail_msg))
    return mod


# first check the python version
pyversion = Version(python_version())

if pyversion >= Version("3.12.10"):
    print(OK, "Python version is %s" % pyversion)
elif pyversion < Version("3.12.10"):
    print(FAIL, "Python version 3.12.10 is required,"
                " but %s is installed." % pyversion)
else:
    print(FAIL, "Unknown Python version: %s" % pyversion)

    
print()
requirements = {'numpy': "2.2.5", 'matplotlib': "3.10.1",'sklearn': "1.6.1", 
                'pandas': "2.2.3",'xgboost': "3.0.0", 'shap': "0.47.2", 
                'polars': "1.27.1", 'seaborn': "0.13.2"}

# now the dependencies
for lib, required_version in list(requirements.items()):
    import_version(lib, required_version)

[ OK ] Python version is 3.12.10

[ OK ] numpy version 2.2.5 is installed.
[ OK ] matplotlib version 3.10.1 is installed.
[ OK ] sklearn version 1.6.1 is installed.
[ OK ] pandas version 2.2.3 is installed.
[ OK ] xgboost version 3.0.0 is installed.
[ OK ] shap version 0.47.2 is installed.
[ OK ] polars version 1.27.1 is installed.
[ OK ] seaborn version 0.13.2 is installed.


### First view of Previous Applicaton Dataset

In [2]:
import pandas as pd
import matplotlib
import sklearn
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

# read in the dataset
df_previous = pd.read_csv('../data/previous_application.csv')

# First view of the dataset
print("Example of the previous_application dataset:")
print(df_previous.head())
print("\n Shape of the previous_application dataset:")
print(df_previous.shape)

Example of the previous_application dataset:
   SK_ID_PREV  SK_ID_CURR NAME_CONTRACT_TYPE  AMT_ANNUITY  AMT_APPLICATION  AMT_CREDIT  AMT_DOWN_PAYMENT  \
0     2030495      271877     Consumer loans     1730.430          17145.0     17145.0               0.0   
1     2802425      108129         Cash loans    25188.615         607500.0    679671.0               NaN   
2     2523466      122040         Cash loans    15060.735         112500.0    136444.5               NaN   
3     2819243      176158         Cash loans    47041.335         450000.0    470790.0               NaN   
4     1784265      202054         Cash loans    31924.395         337500.0    404055.0               NaN   

   AMT_GOODS_PRICE WEEKDAY_APPR_PROCESS_START  HOUR_APPR_PROCESS_START FLAG_LAST_APPL_PER_CONTRACT  \
0          17145.0                   SATURDAY                       15                           Y   
1         607500.0                   THURSDAY                       11                           Y   


##### Columns in Previous Application Dataset

In [10]:
# First view of the columns
print("Basic Information of the previous_application dataset:")
print(df_previous.info())
print("\n\nColumn Names:")
print(df_previous.columns.tolist())

Basic Information of the previous_application dataset:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1670214 entries, 0 to 1670213
Data columns (total 37 columns):
 #   Column                       Non-Null Count    Dtype  
---  ------                       --------------    -----  
 0   SK_ID_PREV                   1670214 non-null  int64  
 1   SK_ID_CURR                   1670214 non-null  int64  
 2   NAME_CONTRACT_TYPE           1670214 non-null  object 
 3   AMT_ANNUITY                  1297979 non-null  float64
 4   AMT_APPLICATION              1670214 non-null  float64
 5   AMT_CREDIT                   1670213 non-null  float64
 6   AMT_DOWN_PAYMENT             774370 non-null   float64
 7   AMT_GOODS_PRICE              1284699 non-null  float64
 8   WEEKDAY_APPR_PROCESS_START   1670214 non-null  object 
 9   HOUR_APPR_PROCESS_START      1670214 non-null  int64  
 10  FLAG_LAST_APPL_PER_CONTRACT  1670214 non-null  object 
 11  NFLAG_LAST_APPL_IN_DAY       1670214 non-null  

##### Coverage of Applicants who have previous record

In [8]:
# Counts of customers in application_data
df_application = pd.read_csv('../data/application_data.csv')
total_customers = set(df_application['SK_ID_CURR'])

prev_customers = set(df_previous['SK_ID_CURR'])

common_customers = total_customers.intersection(prev_customers)

num_total_customers  = len(total_customers )
num_common_customers = len(common_customers)
coverage_percentage = num_common_customers / num_total_customers

print(f"application_data: {num_total_customers} ")
print(f"previous_application: {num_common_customers} ")
print(f"Percentage of customers who have previous record: {coverage_percentage:.2%}")

application_data: 307511 
previous_application: 291057 
Percentage of customers who have previous record: 94.65%


### Aggregate Features of previous_application.csv

In [18]:
continuous_features = {
    'AMT_ANNUITY': ['mean', 'sum'],
    'AMT_APPLICATION': ['mean', 'sum'],
    'AMT_CREDIT': ['mean','sum'],
    'DAYS_DECISION': ['mean'],
    'CNT_PAYMENT': ['mean']
}

categorical_features = {
    'NAME_CONTRACT_TYPE': ['count'],
    'CODE_REJECT_REASON': ['nunique'],
    'NAME_CASH_LOAN_PURPOSE': ['nunique']
}


prev_continuous_agg = df_previous.groupby('SK_ID_CURR').agg(continuous_features)
prev_categorical_agg = df_previous.groupby('SK_ID_CURR').agg(categorical_features)


prev_continuous_agg.columns = pd.Index(['PREV_' + e[0] + "_" + e[1].upper() for e in prev_continuous_agg.columns.tolist()])
prev_categorical_agg.columns = pd.Index(['PREV_' + e[0] + "_" + e[1].upper() for e in prev_categorical_agg.columns.tolist()])
prev_categorical_agg.rename(columns={'PREV_NAME_CONTRACT_TYPE_COUNT': 'PREVIOUS_APPLICATION_COUNTS'}, inplace=True)

In [19]:
prev_status_dummies = pd.get_dummies(df_previous[['SK_ID_CURR', 'NAME_CONTRACT_STATUS']], 
                                     prefix='PREV_STATUS')
prev_status_agg = prev_status_dummies.groupby('SK_ID_CURR').sum()

prev_status_agg.columns = [col.upper() for col in prev_status_agg.columns]

print(prev_status_agg.head())

            PREV_STATUS_APPROVED  PREV_STATUS_CANCELED  PREV_STATUS_REFUSED  PREV_STATUS_UNUSED OFFER
SK_ID_CURR                                                                                           
100001                         1                     0                    0                         0
100002                         1                     0                    0                         0
100003                         3                     0                    0                         0
100004                         1                     0                    0                         0
100005                         1                     1                    0                         0


In [20]:
prev_agg = pd.concat([prev_continuous_agg, prev_categorical_agg, prev_status_agg], axis=1).reset_index()

print("Aggregated features have been concated!")
print("prev_agg head: \n",prev_agg.head())

output_path = '../data/previous_agg_features.csv'

prev_agg.to_csv(output_path, index=False)

Aggregated features have been concated!
prev_agg head: 
    SK_ID_CURR  PREV_AMT_ANNUITY_MEAN  PREV_AMT_ANNUITY_SUM  PREV_AMT_APPLICATION_MEAN  PREV_AMT_APPLICATION_SUM  \
0      100001               3951.000              3951.000                   24835.50                   24835.5   
1      100002               9251.775              9251.775                  179055.00                  179055.0   
2      100003              56553.990            169661.970                  435436.50                 1306309.5   
3      100004               5357.250              5357.250                   24282.00                   24282.0   
4      100005               4813.200              4813.200                   22308.75                   44617.5   

   PREV_AMT_CREDIT_MEAN  PREV_AMT_CREDIT_SUM  PREV_DAYS_DECISION_MEAN  PREV_CNT_PAYMENT_MEAN  \
0              23787.00              23787.0                  -1740.0                    8.0   
1             179055.00             179055.0                

### Try to Merge on Sample Data

In [22]:
df_application = pd.read_csv('../data/application_data.csv')
df_sample = df_application.sample(frac=0.02, random_state=42)

print(f"The 2% data points from the full set -- Sample Shape: {df_sample.shape}")


sample_data_path = '../data/temp_data/sample_application.csv'
df_sample.to_csv(sample_data_path, index=False)
print("\nThe sample data saved")

The 2% data points from the full set -- Sample Shape: (6150, 122)

The sample data saved


In [ ]:
df_sample = pd.read_csv("../data/sample_application.csv")

df_merged_prev_sample = pd.merge(df_sample, prev_agg, on='SK_ID_CURR', how='left')


print("Original Sample Shape:", df_sample.shape)
print("Merged Sample Shape:", df_merged_prev_sample.shape)
print("df_merged_sample head: \n", df_merged_prev_sample.head())

output_path = '../data/temp_data/merged_prev_sample.csv'

df_merged_prev_sample.to_csv(output_path, index=False)

Original Sample Shape: (6150, 122)
Merged Sample Shape: (6150, 137)
df_merged_sample head: 
    SK_ID_CURR  TARGET NAME_CONTRACT_TYPE CODE_GENDER FLAG_OWN_CAR FLAG_OWN_REALTY  CNT_CHILDREN  AMT_INCOME_TOTAL  \
0      384575       0         Cash loans           M            Y               N             2          207000.0   
1      214010       0         Cash loans           F            Y               Y             0          247500.0   
2      142232       0         Cash loans           F            Y               N             0          202500.0   
3      389171       0         Cash loans           F            N               Y             0          247500.0   
4      283617       0         Cash loans           M            N               Y             0          112500.0   

   AMT_CREDIT  AMT_ANNUITY  AMT_GOODS_PRICE NAME_TYPE_SUITE      NAME_INCOME_TYPE            NAME_EDUCATION_TYPE  \
0    465457.5      52641.0         418500.0   Unaccompanied  Commercial associate  Secon

### Now Apply to the Complete Dataset

In [17]:
import pandas as pd
import time

#Record Time for the Full Merge
start_time = time.time()
print("Starts now...")

df_application = pd.read_csv('../data/application_data.csv')

previous_agg = pd.read_csv( '../data/previous_agg_features.csv')


print("\nStarts merging...")
df_merged_full = pd.merge(df_application, prev_agg, on='SK_ID_CURR', how='left')

print("\nStarts filling null values...")
new_agg_features = previous_agg.columns.drop('SK_ID_CURR').tolist()
df_merged_full[new_agg_features] = df_merged_full[new_agg_features].fillna(0)


print("\nCompleted")
print(f"Original application_data Shape: {df_application.shape}")
print(f"Merged application_merged_data Shape: {df_merged_full.shape}")
print("Merged application_merged_data head: \n", df_merged_full.head())


output_path = '../data/application_merged_data.csv'
df_merged_full.to_csv(output_path, index=False)

end_time = time.time()
print(f"\n\n Completed! Time: {end_time - start_time:.2f} s. ")

Starts now...

Starts merging...

Starts filling null values...

Completed
Original application_data Shape: (307511, 122)
Merged application_merged_data Shape: (307511, 137)
Merged application_merged_data head: 
    SK_ID_CURR  TARGET NAME_CONTRACT_TYPE CODE_GENDER FLAG_OWN_CAR FLAG_OWN_REALTY  CNT_CHILDREN  AMT_INCOME_TOTAL  \
0      100002       1         Cash loans           M            N               Y             0          202500.0   
1      100003       0         Cash loans           F            N               N             0          270000.0   
2      100004       0    Revolving loans           M            Y               Y             0           67500.0   
3      100006       0         Cash loans           F            N               Y             0          135000.0   
4      100007       0         Cash loans           M            N               Y             0          121500.0   

   AMT_CREDIT  AMT_ANNUITY  AMT_GOODS_PRICE NAME_TYPE_SUITE NAME_INCOME_TYPE        